In [13]:
import json
import hashlib
import threading
import logging
from datetime import datetime, timezone

credential = "Application-X"
log = logging.getLogger(f"{credential}")
log.setLevel(logging.INFO)

if not log.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
    log.addHandler(handler)

In [9]:

# ── Mini blockchain ledger ────────────────────────────────────────────────

class Ledger:
    """
    Append-only hash-chained credential store.
    Each block: { index, timestamp, key_hash, encrypted, prev_hash, block_hash }
    The chain is valid if every block's prev_hash matches the previous block's block_hash.
    """

    def __init__(self, path: Path):
        self.path   = path
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.chain  = self._load()
        self._lock  = threading.Lock()

    def _load(self) -> list:
        if self.path.exists():
            try:
                chain = json.loads(self.path.read_text())
                if self._verify(chain):
                    log.info(f"Ledger loaded — {len(chain)} block(s), chain valid.")
                    return chain
                else:
                    log.warning("Ledger integrity check FAILED — starting fresh chain.")
            except (json.JSONDecodeError, KeyError):
                log.warning("Ledger corrupt — starting fresh chain.")
        log.info("No existing ledger — initialising genesis block.")
        return [self._genesis()]

    def _genesis(self) -> dict:
        return self._make_block(0, "GENESIS", b"", "0" * 64)

    def _make_block(self, index: int, key_hash: str, encrypted: bytes, prev_hash: str) -> dict:
        block = {
            "index":     index,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "key_hash":  key_hash,
            "encrypted": encrypted.hex() if isinstance(encrypted, bytes) else encrypted,
            "prev_hash": prev_hash,
        }
        block["block_hash"] = self._hash(block)
        return block

    @staticmethod
    def _hash(block: dict) -> str:
        payload = json.dumps({k: v for k, v in block.items() if k != "block_hash"}, sort_keys=True)
        return hashlib.sha256(payload.encode()).hexdigest()

    def _verify(self, chain: list) -> bool:
        for i in range(1, len(chain)):
            prev  = chain[i - 1]
            curr  = chain[i]
            if curr["prev_hash"] != prev["block_hash"]:
                return False
            expected = self._hash({k: v for k, v in curr.items() if k != "block_hash"})
            if curr["block_hash"] != expected:
                return False
        return True

    def append(self, key_hash: str, encrypted: bytes) -> dict:
        with self._lock:
            prev_hash = self.chain[-1]["block_hash"]
            block     = self._make_block(len(self.chain), key_hash, encrypted, prev_hash)
            self.chain.append(block)
            self.path.write_text(json.dumps(self.chain, indent=2))
            log.info(f"Block #{block['index']} committed — hash: {block['block_hash'][:16]}...")
            return block

    def is_valid(self) -> bool:
        return self._verify(self.chain)

In [10]:
# import os
# import json
# import hashlib
# from pathlib import Path
# from cryptography.fernet import Fernet

# # --- test file in current directory ---
# TEST_PATH = Path("./test_ledger.json")

# # --- clean start ---
# if TEST_PATH.exists():
#     TEST_PATH.unlink()


# # --- create ledger ---
# ledger = Ledger(TEST_PATH)

# print("Genesis length:", len(ledger.chain))
# assert len(ledger.chain) == 1
# assert ledger.chain[0]["index"] == 0


# # --- append blocks ---
# block1 = ledger.append("hash1", b"encrypted-data-1")
# block2 = ledger.append("hash2", b"encrypted-data-2")

# print("Chain length:", len(ledger.chain))
# assert len(ledger.chain) == 3


# # --- validate chain ---
# assert ledger.is_valid() is True
# print("Chain valid ✔")


# # --- reload from disk (persistence test) ---
# ledger2 = Ledger(TEST_PATH)

# print("Reloaded length:", len(ledger2.chain))
# assert len(ledger2.chain) == 3
# assert ledger2.is_valid() is True


# # --- tamper test (break integrity) ---
# ledger2.chain[1]["key_hash"] = "corrupted"

# assert ledger2.is_valid() is False
# print("Tamper detection ✔")


# # --- cleanup ---
# if TEST_PATH.exists():
#     TEST_PATH.unlink()

# print("Ledger tests passed ✔")

INFO: No existing ledger — initialising genesis block.
INFO: Block #1 committed — hash: 5c215cfc71f6d796...
INFO: Block #2 committed — hash: 92ff571479d7d61b...
INFO: Ledger loaded — 3 block(s), chain valid.


Genesis length: 1
Chain length: 3
Chain valid ✔
Reloaded length: 3
Tamper detection ✔
Ledger tests passed ✔


In [23]:
import os
import json
import hashlib
import logging
from pathlib import Path

# --- FIX: provide logger ---
log = logging.getLogger(f"{credential}")
log.setLevel(logging.INFO)

if not log.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
    log.addHandler(handler)


# --- test file in current directory ---
BASE_PATH = Path("../module-developer/credential")

# --- clean start ---
if BASE_PATH.exists():
    BASE_PATH.unlink()


# --- create ledger ---
ledger = Ledger(BASE_PATH)

print("Genesis length:", len(ledger.chain))
assert len(ledger.chain) == 1
assert ledger.chain[0]["index"] == 0


# --- append blocks ---
block1 = ledger.append("hash1", b"encrypted-data-1")
block2 = ledger.append("hash2", b"encrypted-data-2")

# print("Chain length:", len(ledger.chain))
# assert len(ledger.chain) == 3


# --- validate chain ---
assert ledger.is_valid() is True
print("Chain valid ✔")


# --- reload from disk ---
ledger2 = Ledger(BASE_PATH)

# print("Reloaded length:", len(ledger2.chain))
# assert len(ledger2.chain) == 3
# assert ledger2.is_valid() is True


# --- tamper test ---
ledger2.chain[1]["key_hash"] = "corrupted"
assert ledger2.is_valid() is False
print("Tamper detection ✔")


# --- cleanup ---
if BASE_PATH.exists():
    BASE_PATH.unlink()

print("Ledger tests passed ✔")

INFO: No existing ledger — initialising genesis block.
INFO: Block #1 committed — hash: 7bfd5a2284143408...
INFO: Block #2 committed — hash: fa9e389258f19613...
INFO: Ledger loaded — 3 block(s), chain valid.


Genesis length: 1
Chain valid ✔
Tamper detection ✔
Ledger tests passed ✔


In [14]:
log = logging.getLogger(__name__)
if not log.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
    log.addHandler(handler)

In [22]:
# """
# credential_chain.py
# ====================
# Ties the Ledger blockchain to the credential file lifecycle.

# Node sequence
# ─────────────
#   Node 0  (genesis)  – hash of the empty credential file
#   Node 1             – hash of file after datetime header is written
#   Node 2             – hash of file after first encrypted credential is appended
#   Node N             – hash of file after every subsequent access / append

# All your existing classes (Credential, EncryptedCredential, Ledger,
# create_credential, format_credential, save_credential, load_credential)
# are imported from wherever they live in your module tree.
# Adjust the import paths to match your project layout.
# """

# import hashlib
# import logging
# from datetime import datetime, timezone
# from pathlib import Path

# # ── adjust these imports to your actual module paths ───────────────────────────
# # from module_developer.credential import (
# #     Credential,
# #     EncryptedCredential,
# #     create_credential,
# #     format_credential,
# #     save_credential,
# #     load_credential,
# # )
# # from module-developer.ledger import Ledger 
# # # your blockchain class
# # from Ledger.Blockchain import

# # ──────────────────────────────────────────────────────────────────────────────

# log = logging.getLogger(__name__)
# log.setLevel(logging.INFO)
# if not log.handlers:
#     _h = logging.StreamHandler()
#     _h.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
#     log.addHandler(_h)


# # ── helpers ────────────────────────────────────────────────────────────────────

# def _hash_file(path: Path) -> str:
#     """SHA-256 of current file contents (empty file → hash of b'')."""
#     data = path.read_bytes() if path.exists() else b""
#     return hashlib.sha256(data).hexdigest()


# def _now_iso() -> str:
#     return datetime.now(timezone.utc).isoformat()


# # ── core lifecycle ─────────────────────────────────────────────────────────────

# class CredentialChain:
#     """
#     Manages a credential file whose every mutation is recorded as a
#     blockchain node in a paired Ledger.

#     Parameters
#     ----------
#     cred_path   : Path to the credential flat-file  (e.g. Path("credential"))
#     ledger_path : Path to the ledger JSON file       (e.g. Path("credential.ledger"))
#     directory   : directory that holds both files    (default: current dir)
#     """

#     def __init__(
#         self,
#         cred_filename:   str  = "credential",
#         ledger_filename: str  = "credential.ledger",
#         directory:       str  = ".",
#     ):
#         self.dir          = Path(directory)
#         self.cred_path    = self.dir / cred_filename
#         self.ledger_path  = self.dir / ledger_filename

#     # ── public API ─────────────────────────────────────────────────────────────

#     def initialise(self) -> Ledger:
#         """
#         Create a brand-new credential file + ledger from scratch.

#         Step 1 – touch empty file  → Node 0  (genesis, hash of empty file)
#         Step 2 – write datetime    → Node 1  (hash of file with header)

#         Returns the open Ledger.
#         """
#         # ── guard: refuse to overwrite an existing chain ───────────────────
#         if self.ledger_path.exists():
#             raise FileExistsError(
#                 f"Ledger already exists at {self.ledger_path}. "
#                 "Call .access() to append, or delete both files to start fresh."
#             )

#         # ── STEP 1: empty file → genesis node ─────────────────────────────
#         self.cred_path.write_bytes(b"")                          # touch
#         empty_hash = _hash_file(self.cred_path)

#         ledger = Ledger(self.ledger_path)                        # creates genesis block internally
#         # Override genesis key_hash so it reflects the actual file state.
#         # If your Ledger.append() is the only public mutator, add the hash
#         # as the first real append instead (see comment below).
#         #
#         # Option A – if Ledger exposes the genesis block for editing:
#         ledger.chain[0]["key_hash"] = empty_hash
#         ledger._save()          # persist the updated genesis block
#         #
#         # Option B – if genesis is immutable, uncomment this and remove A:
#         # ledger.append(empty_hash, b"")   # Node 0: empty-file snapshot

#         log.info("Node 0 (genesis) – empty file hash: %s", empty_hash[:16])

#         # ── STEP 2: write datetime header → Node 1 ────────────────────────
#         timestamp = _now_iso()
#         self.cred_path.write_text(timestamp + "\n", encoding="utf-8")
#         ts_hash = _hash_file(self.cred_path)
#         ts_bytes = timestamp.encode()

#         ledger.append(ts_hash, ts_bytes)
#         log.info("Node 1 (datetime) – file hash: %s", ts_hash[:16])

#         return ledger

#     def add_credential(
#         self,
#         ledger: Ledger,
#         *,
#         include_special_char: bool = True,
#         group_sizes:    tuple = (8, 3, 5),
#         delimiter:      str   = ":",
#         upper_count:    int   = 1,
#         digit_count:    int   = 0,
#         symbol_count:   int   = 0,
#         upper_position: str   = "first",
#     ) -> tuple[Credential, "EncryptedCredential", Path]:
#         """
#         Generate, encrypt, and append one credential to the file.
#         Adds Node N to the ledger (hash of file *after* the append).

#         Returns (plain_credential, encrypted_credential, saved_path)
#         """
#         # validate the chain before touching the file
#         if not ledger.is_valid():
#             raise RuntimeError("Ledger integrity check failed before credential write.")

#         # ── build credential ───────────────────────────────────────────────
#         cred = Credential(
#             create_credential(
#                 include_special_char=include_special_char,
#                 formatter=format_credential(
#                     group_sizes    = group_sizes,
#                     delimiter      = delimiter,
#                     upper_count    = upper_count,
#                     digit_count    = digit_count,
#                     symbol_count   = symbol_count,
#                     upper_position = upper_position,
#                 ),
#             )
#         )
#         cred_enc = cred.encrypt()

#         # ── append to file ─────────────────────────────────────────────────
#         saved_path = save_credential(
#             credential = cred_enc,
#             filename   = self.cred_path.name,
#             directory  = str(self.dir),
#             append     = True,
#             timestamp  = True,
#         )

#         # ── record new file state as a ledger node ─────────────────────────
#         file_hash  = _hash_file(self.cred_path)
#         enc_bytes  = str(cred_enc).encode()
#         ledger.append(file_hash, enc_bytes)
#         node_index = len(ledger.chain) - 1
#         log.info("Node %d – credential appended, file hash: %s", node_index, file_hash[:16])

#         return cred, cred_enc, saved_path

#     def access(self) -> tuple[Ledger, "EncryptedCredential"]:
#         """
#         Open an existing credential chain, validate it, load the latest
#         credential, and record an access node.

#         Returns (ledger, latest_encrypted_credential)
#         """
#         if not self.ledger_path.exists():
#             raise FileNotFoundError(
#                 f"No ledger at {self.ledger_path}. Call .initialise() first."
#             )

#         ledger = Ledger(self.ledger_path)

#         if not ledger.is_valid():
#             raise RuntimeError(
#                 "Ledger integrity check FAILED on load – file may have been tampered with."
#             )

#         log.info("Chain valid. %d nodes loaded.", len(ledger.chain))

#         # ── load most-recent credential from file ──────────────────────────
#         loaded    = load_credential(
#             filename  = self.cred_path.name,
#             directory = str(self.dir),
#             index     = -1,
#         )
#         recovered = loaded.decrypt()
#         log.info("Loaded credential (decrypted): %s", recovered)

#         # ── record this access as a new node ───────────────────────────────
#         file_hash = _hash_file(self.cred_path)
#         access_ts = _now_iso().encode()
#         ledger.append(file_hash, access_ts)
#         node_index = len(ledger.chain) - 1
#         log.info("Node %d – access recorded, file hash: %s", node_index, file_hash[:16])

#         return ledger, loaded

#     def verify(self) -> bool:
#         """Quick integrity check without side-effects."""
#         if not self.ledger_path.exists():
#             log.warning("No ledger file found.")
#             return False
#         ledger = Ledger(self.ledger_path)
#         valid  = ledger.is_valid()
#         log.info("Chain valid: %s (%d nodes)", valid, len(ledger.chain))
#         return valid


# # ── entry point (mirrors your __main__ block) ──────────────────────────────────

# if __name__ == "__main__":

#     chain = CredentialChain(
#         cred_filename   = "credential",
#         ledger_filename = "credential.ledger",
#         directory       = "../module-developer/credential",
#     )

#     # ── first run: build the chain from scratch ────────────────────────────
#     ledger = chain.initialise()
#     print(f"After init  – chain length : {len(ledger.chain)}")   # expect 2 (nodes 0 & 1)

#     # ── add the first credential (Node 2) ─────────────────────────────────
#     cred, cred_enc, saved_path = chain.add_credential(ledger)
#     print(f"Original    : {cred}")
#     print(f"Encrypted   : {cred_enc}")
#     print(f"Saved to    : {saved_path}")
#     print(f"After write – chain length : {len(ledger.chain)}")   # expect 3

#     # ── simulate a later access (Node 3) ──────────────────────────────────
#     ledger2, latest_enc = chain.access()
#     recovered = latest_enc.decrypt()
#     print(f"Recovered   : {recovered}")
#     print(f"After access – chain length : {len(ledger2.chain)}")  # expect 4

#     # ── integrity check ────────────────────────────────────────────────────
#     assert chain.verify(), "Chain should be valid!"
#     print("All checks passed ✔")

ModuleNotFoundError: No module named 'credential'